## Kaggle Competition: The Gemma4 Good Hackathon
Your mission is to create a solution that addresses a real-world challenge using Gemma 4 models, whether that’s an application that helps millions or a specialized model that could exponentially scale innovation.

#### Link <https://www.kaggle.com/competitions/gemma-4-good-hackathon/overview>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



# 📱 Gemma 4 E2B-it Mobile Simulation Notebook (iPad Optimized)
**Model**: google/gemma-4-E2B-it (Edge / Mobile optimized)

**Purpose**: Simulate real mobile AI app experience on iPad  
**Theme**: HK City Personal Finance Assistant (bills, receipts, MPF, rental, tax)

**How to use on iPad**:
1. Run all cells
2. Click the public Gradio URL
3. Open the link in Safari → feels like a native mobile app

In [2]:
# !pip install git+https://github.com/huggingface/transformers.git 

In [3]:
# !pip install -q accelerate gradio pillow --quiet

In [4]:
# !pip install -q aiohttp nest_asyncio --quiet

In [5]:
# import torch
# from transformers import AutoProcessor, AutoModelForCausalLM
# from PIL import Image
# import gradio as gr
# import time
# print("✅ Packages ready for mobile simulation")

import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import gradio as gr
import time
import torchaudio
import soundfile as sf
import json
from io import BytesIO
from IPython.display import display, Markdown

import aiohttp
import asyncio
import nest_asyncio
from io import BytesIO
from PIL import Image

# Enable async in Jupyter/Kaggle notebook
nest_asyncio.apply()

print("✅ Dependencies installed")

/media/johnsonhk88/Big-Data-Disk7/venv-gemma4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dependencies installed


### Load Gemma 4 E2B-it

In [ ]:
MODEL_ID = "google/gemma-4-E4B-it"

print("Loading Gemma 4 E4B-it... (this may take 1-2 minutes)")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
print(f"✅ Gemma 4 E4B-it loaded successfully on {model.device}")

Loading Gemma 4 E4B-it... (this may take 1-2 minutes)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


### Cell 3: Unified Generation Helper

In [ ]:
def gemma_run(messages, max_new_tokens=1024, temperature=0.7):
    start = time.time()

    # === FIX: Extract all images from messages ===
    images = []
    for msg in messages:
        if isinstance(msg.get("content"), list):
            for part in msg["content"]:
                if part.get("type") == "image" and "image" in part:
                    images.append(part["image"])

    
    # FIXED: apply_chat_template returns string → then processor(text=...)
    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    print(f"Prompt: {prompt}")
    
    # === FIX: Pass images= explicitly ===
    inputs = processor(
        text=prompt,
        images=images if images else None,
        return_tensors="pt",
        padding=True
    ).to(model.device)
    
    input_len = inputs["input_ids"].shape[-1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.95
    )
    
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    latency = round(time.time() - start, 2)
    return response, latency

### Text Chat + Reasoning + Thinking Mode

In [ ]:
%%time
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use thinking mode when needed."},
    {"role": "user", "content": "Think step by step: A person in Hong Kong earns HK$45,000/month. How can they maximize tax savings in 2026 with MPF voluntary contributions and self-education expenses?"}
]

response, latency = gemma_run(messages)
print("=== Text + Thinking Test ===\n")
print(response)
print(f"\n⏱️ Latency: {latency}s")

In [ ]:
Markdown(response)

### Image Analysis

In [ ]:
# Upload your image first (bill, receipt, photo) via "Add Data" or Kaggle input
# Example path (change to your uploaded file)
# image = Image.open("/kaggle/input/your-image/example-bill.jpg").convert("RGB")
#
img_url1 = "https://picsum.photos/id/237/200/300"
img_url2 = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/GoldenGate.png"

In [ ]:
# Helper to load image from URL
async def load_image_from_url_async(url: str) -> Image.Image:
    """Download image using aiohttp (async)"""
    async with aiohttp.ClientSession() as session:
        async with session.get(url, timeout=15) as resp:
            resp.raise_for_status()
            data = await resp.read()
            return Image.open(BytesIO(data)).convert("RGB")

# Sync wrapper so you can call it easily in normal cells
def load_image_from_url(url: str) -> Image.Image:
    return asyncio.run(load_image_from_url_async(url))

In [ ]:
image =  load_image_from_url(img_url1)
image

In [ ]:
messages1 = [
    # {"role": "system", "content": "You are a document analyst."},
    {
        "role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What is shown in this image?"}
        ]
    }
]

In [ ]:
messages1

In [ ]:
%%time
response, latency = gemma_run(messages1)   # Note: for multimodal, we will improve in next version if needed
print("=== Image Analysis Test ===\n")
print(response)
print(f"\n⏱️ Latency: {latency}s")

## 3. Video Analysis (via multiple frames)